# Modellazione dei Fattori Latenti di Rischio di Credito con PROC HPPLS

## Sintesi Esecutiva

Una banca al dettaglio vuole prevedere tre esiti di rischio di credito correlati — utilizzo della carta, traiettoria del rapporto debito/reddito e un proxy di probabilità di default — a partire da un ampio insieme di predittori altamente collineari di tipo bureau e macroeconomico. La regressione ordinaria fallisce sotto questa collinearità, quindi questo notebook usa **PROC HPPLS** (partial least squares ad alte prestazioni) per estrarre alcuni fattori latenti che spiegano congiuntamente i predittori e tutti e tre gli esiti, e poi convalida il numero di fattori con un test di convalida incrociata di van der Voet su un segmento di portafoglio tenuto da parte.

## Fonti dei Dati

Tutti i dati sono generati sinteticamente all'interno del notebook con `call streaminit(20240531)` — nessun file esterno né accesso alla rete.

| Dataset | Righe | Variabile | Ruolo | Descrizione |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Chiave cliente univoca portata nell'output punteggiato |
| | | `Segment` | Predittore CLASS | Segmento di portafoglio: Retail, Affluent, Business (valori mantenuti in ASCII: il livello CLASS di riferimento nel modello PLS dipende dall'ordine alfabetico dei valori, quindi tradurli sposterebbe il livello di riferimento e cambierebbe i numeri del modello rispetto alla versione inglese) |
| | | `b1`–`b12` | Predittori | 12 segnali comportamentali mensili di tipo bureau collineari |
| | | `RatePctChg` | Predittore | Esposizione a livello cliente alla variazione del tasso di interesse |
| | | `InquiryCount` | Predittore | Conteggio delle recenti richieste di credito (hard inquiry) |
| | | `Utilization` | Risposta | Utilizzo del credito rotativo (%) |
| | | `DTIChange` | Risposta | Variazione del rapporto debito/reddito |
| | | `DefaultProp` | Risposta | Proxy della probabilità di default (0–1) |
| | | `Role` | Partizione | Flag di convalida TRAIN (~75%) / TEST (~25%) |

# Modellazione dei Fattori Latenti di Rischio di Credito con PROC HPPLS

Le banche affrontano regolarmente il problema del **predittore ampio e collineare**: decine di attributi bureau mensili, esposizioni macroeconomiche e segnali comportamentali che si muovono insieme, usati per prevedere diversi esiti di rischio che sono a loro volta correlati. I minimi quadrati ordinari sono instabili qui perché la matrice dei predittori è quasi singolare.

I **minimi quadrati parziali (PLS)** risolvono questo problema estraendo un numero ridotto di fattori latenti dalla *covarianza incrociata* dei predittori (X) e delle risposte (Y), in modo che i fattori siano calibrati per prevedere gli esiti — non solo per riassumere X. `PROC HPPLS` è la controparte ad alte prestazioni di `PROC PLS`, e aggiunge esecuzione multithread, partizionamento dei dati per la convalida e il test di randomizzazione di van der Voet per scegliere il numero di fattori.

In questo notebook costruiamo un unico modello PLS che prevede simultaneamente tre esiti di rischio di credito correlati e usiamo un segmento di convalida tenuto da parte per confermare quanti fattori latenti i dati supportino realmente.

## Passo 1 — Generare un pannello sintetico di rischio di credito

Simuliamo 600 clienti. Due driver latenti non osservati (`stress` e `tenure`) generano dodici segnali bureau collineari `b1`–`b12` più le esposizioni di tasso e di richiesta — esattamente la struttura ad alta correlazione per cui PLS è progettato. Le tre risposte (`Utilization`, `DTIChange`, `DefaultProp`) sono diverse combinazioni lineari degli stessi driver, quindi anche loro sono correlate. Un flag `Role` tiene da parte circa un quarto del portafoglio come segmento di convalida.

In [1]:
DATI credit;
   CHIAMARE streaminit(20240531);
   LUNGHEZZA Segment $8 Role $5;
   FARE CustomerID = 1 FINO_A 600;
      /* segmento del cliente (predittore categoriale) -- valori tenuti in ASCII:
         il livello CLASS di riferimento nel modello PLS dipende dall'ordine
         alfabetico dei valori, quindi tradurli sposterebbe il riferimento e
         cambierebbe i numeri del modello rispetto alla versione inglese */
      u = rand('uniform');
      SE_COND u < 0.34 ALLORA Segment = 'Retail';
      ALTRIMENTI SE_COND u < 0.70 ALLORA Segment = 'Affluent';
      ALTRIMENTI Segment = 'Business';

      /* driver macro / comportamentali non osservati (collineari) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 predittori mensili collineari di tipo bureau b1-b12 */
      VETTORE b{12} b1-b12;
      FARE j = 1 FINO_A 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      FINE;

      /* covariate macro, anch'esse legate ai driver */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tre esiti di rischio di credito correlati */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      SE_COND DefaultProp < 0 ALLORA DefaultProp = 0;

      /* tiene da parte ~25% come segmento di convalida */
      SE_COND rand('uniform') < 0.25 ALLORA Role = 'TEST';
      ALTRIMENTI Role = 'TRAIN';

      USCITA;
   FINE;
   RIMUOVERE u stress tenure j;
ESEGUIRE;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.44 seconds
  cpu   0.44 seconds


## Passo 2 — Adattare il modello PLS multi-risposta con convalida incrociata

La chiamata principale mostra le istruzioni e le opzioni fondamentali di `PROC HPPLS` che un modellatore di rischio userebbe davvero:

- **`MODEL`** elenca tutte e tre le risposte a sinistra e l'intero insieme di predittori collineari a destra; **`/ solution`** stampa i coefficienti predittivi finali sulla scala grezza.
- **`CLASS Segment`** introduce il segmento di portafoglio come predittore categoriale (deve precedere `MODEL`).
- **`ID CustomerID`** porta la chiave cliente nell'output punteggiato.
- **`CVTEST(stat=press ...)`** esegue il test di randomizzazione di van der Voet per scegliere il numero di fattori in modo oggettivo anziché a occhio; `seed=` lo rende riproducibile.
- **`PARTITION rolevar=Role(...)`** adatta e punteggia sulle righe di addestramento e tiene da parte il segmento `TEST` in modo che il PRESS di convalida incrociata sia misurato fuori campione.
- **`VARSS`** e **`DETAILS`** riportano quanta variazione di X e di Y ciascun fattore successivo spiega.
- **`OUTPUT`** scrive i valori previsti, i residui, i punteggi X e il PRESS per le righe adattate (di addestramento) in un dataset punteggiato, indicizzato per `CustomerID`.

In [2]:
PROCEDURA hppls DATI=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   CLASSE Segment;
   id CustomerID;
   MODELLO Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   ETICHETTA Utilization="Utilizzo Credito (%)" DTIChange="Variazione Rapporto Debito/Reddito"
         DefaultProp="Probabilità di Default (proxy)" Segment="Segmento di Portafoglio"
         RatePctChg="Variazione % Tasso" InquiryCount="Numero Richieste di Credito";
   partition rolevar=Role(train='TRAIN' test='TEST');
   USCITA out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
ESEGUIRE;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segmento di Portafog: 3 levels: Affluent Business Retail

Response Variable(s): Utilizzo Credito (%) Variazione Rapporto  Probabilità di Defau
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Variazione % Tasso Numero Richieste di  Segmento di Portafog

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8    


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Passo 3 — Riassumere il profilo di rischio previsto

Con il modello adattato, profiliamo gli esiti previsti sull'intero portafoglio. `PROC MEANS` riporta la tendenza centrale e la dispersione di ciascuna risposta prevista in modo che il team di rischio possa verificare la scala (ad es. utilizzo previsto centrato a metà dei 40, proxy di default vicino al tasso di base assunto dell'8%).

In [3]:
PROCEDURA MEDIE DATI=scored mean std MIN MAX maxdec=3;
   VARIABILE Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   ETICHETTA Pred_Utilization="Utilizzo Previsto (%)" Pred_DTIChange="Variazione D/R Prevista"
         Pred_DefaultProp="Probabilità di Default Prevista";
ESEGUIRE;

                                                  The MEANS Procedure

 Variable          Label                                      Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Utilizzo Previsto (%)                    47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Variazione D/R Prevista                   0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Probabilità di Default Prevista           0.090       0.049       0.008       0.235
 -----------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Passo 4 — Ispezionare i singoli clienti punteggiati

Infine elenchiamo alcuni clienti dal segmento **di addestramento** adattato con il loro flag `Role` originale, l'utilizzo previsto e il residuo. (Le righe `TEST` tenute da parte non sono deliberatamente punteggiate, quindi filtriamo per `Role='TRAIN'` per mostrare righe completamente popolate.) Questo è il tipo di output a livello di riga che un analista allegherebbe a un rapporto di monitoraggio del modello o alimenterebbe a valle in un motore di definizione dei limiti.

In [4]:
PROCEDURA STAMPARE DATI=scored(obs=8) ETICHETTA;
   DOVE Role = 'TRAIN';
   VARIABILE CustomerID Role Pred_Utilization Resid_Utilization;
   ETICHETTA CustomerID="ID Cliente" Role="Ruolo" Pred_Utilization="Utilizzo Previsto (%)"
         Resid_Utilization="Residuo Utilizzo";
ESEGUIRE;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretare i risultati

La tabella **Percent Variation Accounted for** mostra che il primo fattore da solo assorbe circa tre quarti della variazione dei predittori (74,07%, la direzione "stress" collineare dominante), mentre il secondo e il terzo fattore sono dove viene spiegata la maggior parte della variazione della *risposta* (37,94% e 10,46%, contro solo 25,83% dal primo) — il tratto distintivo del PLS che riorienta le componenti verso la previsione anziché la pura varianza di X. A otto fattori gli R-quadrati per risposta si assestano a 0,81 (Utilization), 0,78 (DTIChange) e 0,74 (DefaultProp), confermando che i tre esiti di rischio di credito sono ben catturati da una struttura latente a bassa dimensionalità.

Il test di **convalida incrociata PRESS di van der Voet** è il fattore decisivo qui: il PRESS sul segmento tenuto da parte scende bruscamente attraverso i primi quattro fattori (8816 → 4772 → 3318 → 3244) e poi si appiattisce e risale leggermente, quindi il test seleziona **quattro fattori** come modello parsimonioso. Estrarne di più inseguirebbe il rumore nell'ampio blocco bureau e degraderebbe le prestazioni fuori campione — esattamente l'overfitting che un modello di rischio di credito deve evitare prima della messa in produzione.

I coefficienti `SOLUTION` danno alla banca uno scorecard lineare interpretabile e regolarizzato sulla scala della variabile originale, con `RatePctChg` (≈0,80, 0,88, 0,63 sui tre esiti) e `InquiryCount` (≈0,47, 0,36, 0,58) che emergono come i driver singoli più forti. In pratica un modellatore ora riadatterebbe con `nfac=4`, monitorerebbe i residui nel dataset `scored`, e promuoverebbe i punteggi fattoriali o i coefficienti in una pipeline di decisione del rischio in produzione.